Empezaremos diagnosticando problemas en el dataset y en qué columnas aparecen. Pretendemos responder:

- ¿Qué columnas parecen problemáticas?
- ¿El problema es solo MVs o también hay tipos incorrectos, duplicados o valores sospechosos?
- ¿Hay columnas donde el problema parece grave y otras donde parece menor?
- ¿Qué tipo de problema tiene cada columna: ausencia de datos, formato extraño, cardinalidad rara, texto sucio, etc.?

In [ ]:
import pandas as pd

df = pd.read_csv('datasets/AB_NYC_2019.csv')

print("Duplicados exactos:", df.duplicated().sum())
print(df.head())


In [ ]:
print(df.info())

In [ ]:
df.describe(include='all')

A primera vista, el dataset presenta posibles problemas en columnas como ***name, host_name, last_review y reviews_per_month***, que parecen tener valores ausentes. Además, last_review probablemente requiere conversión a fecha, y algunas variables numéricas como price o minimum_nights podrían contener valores extremos que conviene revisar. No se han tomado todavía decisiones de limpieza; este paso solo identifica áreas problemáticas. El siguiente paso es medir cuántos nulos hay:

- ¿Qué columnas tienen nulos?
- ¿Cuántos nulos tiene cada una? ¿Qué porcentaje representan sobre el total?

In [ ]:
null_summary = pd.DataFrame({
    "n_nulls": df.isnull().sum(),
    "pct_nulls": df.isnull().mean() * 100
}).sort_values("pct_nulls", ascending=False)

null_summary[null_summary['n_nulls'] > 0]

Las variables ***last_review y reviews_per_month*** no parecen presentar un ***missing aleatorio*** sin más, sino una ausencia posiblemente semántica: es plausible que indiquen alojamientos sin reseñas recientes o sin historial de reseñas. En cambio, ***host_name y name*** son variables textuales con una cantidad mínima de nulos, por lo que su problema parece menor y más cercano a incompletitud puntual que a una ausencia estructural de información.

In [ ]:
df['name'] = df['name'].fillna('Unknown')
df['host_name'] = df['host_name'].fillna('Unknown')
df['last_review'] = pd.to_datetime(df['last_review'], errors="coerce")

Dado que ***name y host_name*** presentan una cantidad mínima de valores ausentes y son variables textuales secundarias, se imputan con ***"Unknown"*** para evitar pérdida innecesaria de filas.
La variable last_review se convierte a formato fecha y se conservan sus nulos, ya que parecen reflejar ausencia de reseñas más que un error aleatorio de captura.
En ***reviews_per_month***, los nulos se interpretan con cautela, pueden dejarse como ausentes o imputarse a 0 si se asume que representan listings sin actividad de reseñas.

Por último, puede realizarse una comprobación adicional de coherencia interna entre variables. En este punto, no parece prudente seguir modificando el dataset sin tener claro el objetivo posterior del análisis o del pipeline. Antes de aplicar nuevas transformaciones, conviene verificar si ciertos valores aparentemente problemáticos responden en realidad a situaciones plausibles dentro del dominio. Algunos ejemplos serían:

- Comprobar si una disponibilidad anual igual a 0 se asocia realmente con listings sin actividad reciente o con calendarios cerrados.
- Comprobar si existen listings con un número de reseñas positivo pero sin actividad reciente en variables como ***reviews_per_month o last_review***.
- Analizar si ciertas combinaciones de variables reflejan inconsistencias reales o simplemente casos límite válidos.

In [ ]:
(df["availability_365"] == 0).sum()
df.loc[df["availability_365"] == 0, [
    "availability_365",
    "number_of_reviews",
    "reviews_per_month",
    "last_review",
    "price",
    "minimum_nights"
]].head()

((df["calculated_host_listings_count"] == 0) & (df["number_of_reviews"] > 0)).sum()

Como cierre del EDA, se incluye un mapa de calor de correlaciones entre variables numéricas para identificar relaciones lineales potencialmente relevantes. Este análisis permite detectar asociaciones iniciales entre variables, aunque no implica causalidad ni sustituye una revisión individual más detallada de cada caso.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

relevant_cols = [
    "price",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "calculated_host_listings_count",
    "latitude",
    "longitude",
    "availability_365"
]

corr_df = df[relevant_cols].copy()
corr_matrix = corr_df.corr(numeric_only=True)

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(9, 7))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5
)

plt.title("Correlación entre variables numéricas relevantes")
plt.tight_layout()
plt.show()